In [ ]:
# Install the PySR library and its dependencies
!pip install -Uq pysr dask[complete]

# Install the Julia backend (This may take 2-3 minutes)
import pysr
pysr.install()

[juliapkg] Found dependencies: /usr/local/lib/python3.12/dist-packages/pysr/juliapkg.json
[juliapkg] Found dependencies: /usr/local/lib/python3.12/dist-packages/juliapkg/juliapkg.json
[juliapkg] Found dependencies: /usr/local/lib/python3.12/dist-packages/juliacall/juliapkg.json
[juliapkg] Locating Julia 1.10.3 - 1.11
[juliapkg] Using Julia 1.11.5 at /usr/local/bin/julia
[juliapkg] Using Julia project at /root/.julia/environments/pyjuliapkg
[juliapkg] Writing Project.toml:
           | [deps]
           | SymbolicRegression = "8254be44-1295-4e6a-a16d-46603ac705cb"
           | Serialization = "9e88b42a-f829-5b0c-bbe9-9e923198166b"
           | PythonCall = "6099a3de-0909-46bc-b1f4-468b9a2dfc0d"
           | OpenSSL_jll = "458c3c95-2e84-50aa-8efc-19380b2a3a95"
           | 
           | [compat]
           | SymbolicRegression = "~1.11"
           | Serialization = "^1"
           | PythonCall = "=0.9.26"
           | OpenSSL_jll = "~3.0"
[juliapkg] Installing packages:
           | impo

/usr/local/lib/python3.12/dist-packages/pysr/deprecated.py:10: FutureWarning: The `install` function has been removed. PySR now uses the `juliacall` package to install its dependencies automatically at import time. 
  warnings.warn(


In [ ]:
import dask.dataframe as dd
import pandas as pd
import numpy as np

# Load your Kaggle dataset (replace with your path)
df = dd.read_csv('/content/vmCloud_data.csv')

# --- MOCK DATA FOR 1M ROWS (Structure provided by you) ---
n_rows = 1_000_000
data = {
    'cpu_usage': np.random.uniform(5, 95, n_rows),
    'memory_usage': np.random.uniform(10, 90, n_rows),
    'network_traffic': np.random.uniform(50, 5000, n_rows),
    'num_executed_instructions': np.random.uniform(1e5, 1e9, n_rows),
    'energy_efficiency': np.random.uniform(0.1, 0.9, n_rows) # Target for Scaling
}
df = dd.from_pandas(pd.DataFrame(data), npartitions=10)

# Preprocessing: Efficiency is often a ratio, so we add a derived feature
df['inst_per_cpu'] = df['num_executed_instructions'] / (df['cpu_usage'] + 1e-6)

# Sample 50k rows for formula discovery (Statistically significant for 1M)
train_df = df.sample(frac=0.05).compute()

X = train_df[['cpu_usage', 'memory_usage', 'network_traffic', 'inst_per_cpu']]
y = train_df['energy_efficiency']

In [ ]:
from pysr import PySRRegressor

# Configure the search for the formula
model = PySRRegressor(
    niterations=50,             # How many cycles to evolve the math
    binary_operators=["+", "*", "/", "-"],
    unary_operators=["inv(x) = 1/x", "log", "sqrt", "exp"],
    constraints={'/': (-1, 1)}, # Prevents division by zero errors
    nested_constraints={"log": {"log": 0}, "exp": {"exp": 0}}, # Simple formulas only
    model_selection="best",     # Balances accuracy vs complexity
    loss="loss(prediction, target) = (prediction - target)^2",
    extra_sympy_mappings={'inv': lambda x: 1/x} # Added mapping for custom inv operator
)

# This will run the Julia search engine
model.fit(X, y)

# Print the final result
print("\n" + "="*40)
print("BEST SCALING FORMULA DISCOVERED:")
print(model.get_best().equation)
print("="*40)

/usr/local/lib/python3.12/dist-packages/pysr/sr.py:1036: FutureWarning: `loss` has been renamed to `elementwise_loss` in PySRRegressor. Please use that instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/pysr/sr.py:2811: UserWarning: Note: it looks like you are running in Jupyter. The progress bar will be turned off.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/pysr/sr.py:2265: UserWarning: Note: you are running with more than 10,000 datapoints. You should consider turning on batching (https://ai.damtp.cam.ac.uk/pysr/options/#batching). You should also reconsider if you need that many datapoints. Unless you have a large amount of noise (in which case you should smooth your dataset first), generally < 10,000 datapoints is enough to find a functional form with symbolic regression. More datapoints will lower the search speed.
  warnings.warn(
Compiling Julia backend...
INFO:pysr.sr:Compiling Julia backend...
[ Info: Note: you are running with more than 10,000 dat


Expressions evaluated per second: 0.000e+00
Progress: 0 / 1550 total iterations (0.000%)
════════════════════════════════════════════════════════════════════════════════════════════════════
───────────────────────────────────────────────────────────────────────────────────────────────────
Complexity  Loss       Score      Equation
───────────────────────────────────────────────────────────────────────────────────────────────────
════════════════════════════════════════════════════════════════════════════════════════════════════
Press 'q' and then <enter> to stop execution early.

Expressions evaluated per second: 1.310e+01
Progress: 1 / 1550 total iterations (0.065%)
════════════════════════════════════════════════════════════════════════════════════════════════════
───────────────────────────────────────────────────────────────────────────────────────────────────
Complexity  Loss       Score      Equation
1           5.311e-02  0.000e+00  y = 0.49957
─────────────────────────────────